In [6]:
import datetime 
from dateutil.relativedelta import relativedelta
import pandas as pd
import numpy as np
import pymssql
from shutil import copyfile
from openpyxl import load_workbook
import os
from google.cloud import bigquery
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = 'BQ.json'

DB_info = {'server':'192.168.61.119:7622', 'user':'BAReporting', 'password':'KeHeCReme8he'}

client = bigquery.Client()

In [ ]:
sql = f"""
SELECT 
    Time,
    DeviceId,
    UserId,    
    Platform,
WHERE lower(EventLabel) like 'hk.lms2.35336.tab.-990.1'
GROUP BY Time, DeviceId, UserId, Platform

FROM `openrice-production.ORGA.PV_20260914` 
"""

#Execute query
df_big_query = client.query(sql).result().to_dataframe()
df_big_query

In [ ]:
#Query BQ
sql = f"""
SELECT 
    extract(month from time) as month,
    Platform,
    DeviceId,
    userid,
FROM `openrice-production.ORGA.PV_20260914` 
WHERE lower(EventLabel) like 'hk.lms2.35336.tab.-990.1'
GROUP BY userid, DeviceId, Platform, month,
"""

#Execute query
df_big_query = client.query(sql).result().to_dataframe()
df_big_query

In [ ]:
# Daily Brand Page (LMS) Pageview
sql = """
SELECT
  DATE(Time, 'Asia/Hong_Kong') AS report_date,
  Platform,
  COUNT(*) AS screenview_count,
  COUNT(DISTINCT NULLIF(TRIM(UserId), '')) AS unique_users,
  COUNT(DISTINCT NULLIF(TRIM(DeviceId), '')) AS unique_devices
FROM `openrice-production.ORGA.SV_20260916`
WHERE LOWER(EventLabel) = 'hk.lms2.35336.tab.-990.1'
GROUP BY 1, 2
ORDER BY 1, 2;
"""

In [ ]:
# Daily Theme Listing Impressions --> adv search
sql = """
SELECT
  DATE(Time, 'Asia/Hong_Kong') AS report_date,
  Platform,
  COUNT(*) AS advance_search_clicks,
  COUNT(DISTINCT NULLIF(TRIM(UserId), '')) AS unique_users,
  COUNT(DISTINCT NULLIF(TRIM(DeviceId), '')) AS unique_devices
FROM `openrice-production.ORGA.PV_20260916`
WHERE LOWER(EventAction) = 'or.advsearch.open.filter'
  AND EXISTS (
    SELECT 1
    FROM UNNEST(EventLabel.list) AS label
    WHERE LOWER(label.item.Param) = 'sn'
      AND LOWER(label.item.Value) IN (
        'hk.lms2.35336.tab.-990.1',
        'hkhk.lms2.35336.tab.-990.1'
      )
  )
GROUP BY 1, 2
ORDER BY 1, 2;
"""